<div align="center">

<a href="https://colab.research.google.com/github/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/weeks/week-07/NB07_specification.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Week 07: Specification Problems: Reward Hacking, Side Effects and Goodhart's Law

**Machine Ethics and Artificial Intelligence Safety (11118BLG001)**<br>Prof. Dr. Utku Kose<br>Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye<br>Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)<br>[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

*This course is updated in line with current developments in the field. Last update: September 2026.*

</div>

## How to use this notebook

This notebook is a self-contained version of the week: the lecture text comes first, followed by the hands-on lab. Run the cells in order with Shift+Enter. Exercise cells start with `None` placeholders and print feedback through `check(...)`. Reference solutions sit in collapsed cells: open them only after a genuine attempt. The interactive lab and the self-assessment are available at <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-07/lab.html>.

## Overview

An optimiser does what its objective says, not what its designers meant. This week studies the gap between specified and intended objectives, from Goodhart's law and reward hacking to negative side effects, with gridworld experiments that make the failures visible [1, 2, 3].

## Learning outcomes

By the end of the week, students are expected to distinguish the four variants of Goodhart's law, to define reward hacking and explain why it is hard to rule out, to describe empirical evidence that capable agents exploit misspecified rewards more, and to implement and evaluate a side-effect penalty in a gridworld.

## The specification gap

Designers rarely write down exactly what they want. They write a proxy: a reward function, a loss or a metric that is easier to measure. Amodei and colleagues illustrated two resulting problems with a cleaning robot [1]. A robot rewarded for not seeing messes may learn to cover them instead of cleaning them, which is reward hacking. A robot rewarded only for speed may knock over a vase on the shortest route, which is a negative side effect of an objective that says nothing about vases.

Goodhart's law states that a measure used as a target tends to lose its value as a measure. Manheim and Garrabrant distinguished four mechanisms [2]. Regressional Goodhart arises because selecting on a noisy proxy also selects for noise. Extremal Goodhart arises because the relation between proxy and goal can break down in extreme regions never seen before. Causal Goodhart arises when intervening on the proxy does not change the goal. Adversarial Goodhart arises when other agents exploit the proxy. Figure 7.1 simulates the regressional and extremal variants: As selection on a heavy-tailed proxy grows stronger, the true value of the selected items first rises and then falls back.

![Figure 7.1](https://raw.githubusercontent.com/utkukose/machine-ethics-ai-safety-NB-lecture/main/weeks/week-07/figures/w07_fig1.png)

*Figure 7.1. Simulated selection on a proxy equal to the true value plus heavy-tailed noise. Moderate selection raises the true value of the chosen items; extreme selection mainly selects noise [2].*

## Reward hacking in theory and practice

Skalse and colleagues gave reward hacking a formal definition [4]. A proxy reward is unhackable with respect to a set of policies if increasing the proxy return never decreases the true return. They showed that for the set of all stochastic policies, two reward functions can be unhackable only in trivial cases, for example when one of them is constant. Simplified rewards therefore almost always leave room for hacking, and safety must come from how optimisation is constrained and monitored.

Pan, Bhatia and Steinhardt studied misspecified rewards in several reinforcement learning environments [5]. More capable agents, with larger models, more training or finer actions, achieved higher proxy reward and lower true reward. They also observed phase transitions, in which a small increase in capability produced a qualitative change towards exploitative behaviour. Such transitions are hard to anticipate from tests on weaker systems.

## Side effects and impact regularisation

Leike and colleagues released AI safety gridworlds, small environments that isolate safety problems such as avoiding side effects, safe interruptibility, an absent supervisor and reward gaming [3]. Each environment has a visible reward and a hidden performance function that captures what designers actually want, so that an agent can score well while behaving badly.

One response is to penalise impact. Turner, Hadfield-Menell and Tadepalli proposed attainable utility preservation: The agent is penalised for changing its ability to achieve a set of auxiliary goals, compared with doing nothing [6]. Breaking a vase reduces the attainable utility of goals that involve the vase, so a conservative agent avoids it without an explicit vase rule. The lab uses a simpler penalty that is attached directly to the vase and shows the trade-off between the penalty weight and the length of the detour.

## Designing better objectives

Specification problems cannot be solved by better proxies alone. Practical defences combine several layers: learning objectives from human feedback, which Week 11 examines together with its own failure modes; limiting optimisation pressure; adding penalties for irreversible changes; monitoring for behaviour that scores well on the proxy but looks wrong to humans; and keeping humans able to interrupt. Kose and Vasant's fading intelligence theory adds a structural safeguard of a different kind, since it bounds the lifetime over which any single system can accumulate exploitative strategies [7].

> **Pause and reflect.** Name a metric in academic life, such as citation counts or grade averages, that has become a target. Which variant of Goodhart's law best describes what happened?

# Hands-on lab

The notebook trains a tabular Q-learning agent in the corridor gridworld of the lab, compares the learned behaviour under the misspecified and the penalised reward, and reproduces the Goodhart simulation [2, 3].

In [ ]:
# Setup: common libraries and the self-check helper used in the exercises.
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
SEED = 2026

def check(name, value, expected, tol=1e-6):
    """Compare an exercise answer with the reference value and print feedback."""
    if value is None:
        print(f"[{name}] Not attempted yet. Replace None with your answer and run the cell again.")
        return
    try:
        ok = bool(np.allclose(np.asarray(value, dtype=float), np.asarray(expected, dtype=float), atol=tol))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"[{name}] " + ("Correct." if ok else "Not yet. Revisit the explanation above and try again."))

## 1. The corridor gridworld

In [ ]:
GRID = [".......", ".#####.", "S..V..G", ".#####.", "......."]
R, C = len(GRID), len(GRID[0])
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1)]
START, VASE_COST = (2, 0), 3.0

def step_env(state, a):
    r, c = state[0] + MOVES[a][0], state[1] + MOVES[a][1]
    if not (0 <= r < R and 0 <= c < C) or GRID[r][c] == "#":
        r, c = state
    return (r, c), GRID[r][c] == "G"

def reward(state, step_cost, vase_penalty):
    cell = GRID[state[0]][state[1]]
    return (10.0 if cell == "G" else 0.0) - step_cost - (vase_penalty if cell == "V" else 0.0)

def q_learning(step_cost, vase_penalty, episodes=3000, alpha=0.3, eps=0.2, seed=SEED):
    rng = np.random.default_rng(seed)
    Q = np.zeros((R, C, 4))
    for _ in range(episodes):
        s = START
        for _ in range(60):
            a = rng.integers(4) if rng.uniform() < eps else int(np.argmax(Q[s]))
            s2, done = step_env(s, a)
            target = reward(s2, step_cost, vase_penalty) + (0 if done else np.max(Q[s2]))
            Q[s][a] += alpha * (target - Q[s][a])
            s = s2
            if done:
                break
    return Q

def greedy_path(Q):
    s, path = START, [START]
    for _ in range(40):
        s, done = step_env(s, int(np.argmax(Q[s])))
        path.append(s)
        if done:
            break
    return path

def show(path, title):
    img = np.array([[0.0 if ch != "#" else -1.0 for ch in row] for row in GRID])
    for (r, c) in path:
        img[r, c] = 1.0
    plt.figure(figsize=(4.5, 2.2)); plt.imshow(img, cmap="PuBu"); plt.title(title)
    for r in range(R):
        for c in range(C):
            if GRID[r][c] in "SGV":
                plt.text(c, r, GRID[r][c], ha="center", va="center", fontweight="bold")
    plt.axis("off"); plt.show()

Q0 = q_learning(step_cost=0.5, vase_penalty=0.0)
path0 = greedy_path(Q0)
show(path0, "Learned path, reward ignores the vase")

### Your turn, exercise 1

Compute the true return of `path0`, where every step costs 0.5, reaching the goal gives 10 and entering the vase cell costs `VASE_COST`. Do not count the start cell.

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
def _true_return_reference(path, step_cost=0.5):
    return sum(reward(s, step_cost, VASE_COST) for s in path[1:])

In [ ]:
true_return = None  # your computation
check("Exercise 1", true_return, _true_return_reference(path0))

## 2. Adding a side-effect penalty

With step cost c, the detour is four steps longer, so it costs 4c more. The agent should avoid the vase exactly when the penalty in its reward exceeds 4c.

In [ ]:
for pen in [0.0, 1.0, 2.5, 4.0]:
    p = greedy_path(q_learning(0.5, pen))
    broke = any(GRID[r][c] == "V" for r, c in p)
    print(f"penalty {pen:3.1f}: path length {len(p) - 1:2d}, vase broken: {broke}, true return {_true_return_reference(p):.2f}")

### Your turn, exercise 2

For a step cost of 1.0, predict the smallest penalty that makes the agent take the detour, then confirm it with `q_learning`. Is the true return higher or lower on the detour at this step cost? Explain the result in two sentences.

In [ ]:
# Your code here

## 3. Goodhart's law in simulation

In [ ]:
rng = np.random.default_rng(1)
true = rng.normal(0, 1, 200000)
proxy = true + rng.standard_t(2, true.size)
order = np.argsort(-proxy)
qs = np.logspace(-4, 0, 40)
mean_true = [true[order[: max(1, int(q * true.size))]].mean() for q in qs]
plt.plot(np.log10(1 / qs), mean_true, marker=".")
plt.xlabel("log10(1 / fraction selected)"); plt.ylabel("mean true value of the selection"); plt.title("Regressional and extremal Goodhart")
plt.show()

## Self-assessment and reflection

Take the self-assessment in the interactive lab: <https://utkukose.github.io/machine-ethics-ai-safety-NB-lecture/weeks/week-07/lab.html> (tab: Check yourself). Then answer the reflection prompts:

1. Describe a reward or metric in a system you have built or used that could be hacked. What would the hack look like?
2. In the lab, the true return improved only in some settings when the penalty was added. What does that teach about tuning penalties?
3. Which layer of defence from the last section would you prioritise for a recommender system, and why?

## Weekly task and submission

Take an optimisation objective from your research or from a familiar system and write a specification risk analysis of about 400 words: the intended goal, the proxy, one plausible hack, one plausible side effect, the Goodhart variant involved and a mitigation. Attach the notebook with both exercises completed.

The weekly task supports self-learning and builds a personal portfolio. When the course is followed with the instructor during an active semester, the task can be sent together with the exported learning log to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation.

## Research and report assignment (optional)

**A catalogue of specification gaming.** Collect at least ten documented cases of specification gaming or reward hacking from the literature, classify them by the variants of Goodhart's law and identify what a better specification would have required [2, 4].

This research assignment is optional and supports self-learning. When the related weeks are followed within the course during an active semester, the report can be sent to utkukose@sdu.edu.tr or utkukose@gmail.com for evaluation. Unless the assignment states otherwise, a report has 1500 to 2500 words, follows the structure of an academic paper, cites at least six scholarly or official sources in square brackets and ends with a reference list.

## Midterm capstone

This week closes the first half of the course with the [Midterm Capstone: Ethics, Fairness and Specification in Practice](https://github.com/utkukose/machine-ethics-ai-safety-NB-lecture/blob/main/exams/midterm/README.md).

This capstone also supports self-learning and can be completed at any pace. When the course is taught actively in a semester, the midterm capstone is sent by e-mail to utkukose@sdu.edu.tr or utkukose@gmail.com no later than 23:53 (Türkiye time) on the last Sunday of Week 7, with the report and all code files attached or linked.

## References

[1] Amodei, D., Olah, C., Steinhardt, J., Christiano, P., Schulman, J., & Mané, D. (2016). Concrete problems in AI safety. arXiv preprint arXiv:1606.06565. <https://arxiv.org/abs/1606.06565>

[2] Manheim, D., & Garrabrant, S. (2018). Categorizing variants of Goodhart's law. arXiv preprint arXiv:1803.04585. <https://arxiv.org/abs/1803.04585>

[3] Leike, J., Martic, M., Krakovna, V., Ortega, P. A., Everitt, T., Lefrancq, A., Orseau, L., & Legg, S. (2017). AI safety gridworlds. arXiv preprint arXiv:1711.09883. <https://arxiv.org/abs/1711.09883>

[4] Skalse, J., Howe, N. H. R., Krasheninnikov, D., & Krueger, D. (2022). Defining and characterizing reward hacking. In *Advances in Neural Information Processing Systems 35*.

[5] Pan, A., Bhatia, K., & Steinhardt, J. (2022). The effects of reward misspecification: Mapping and mitigating misaligned models. In *International Conference on Learning Representations (ICLR)*.

[6] Turner, A. M., Hadfield-Menell, D., & Tadepalli, P. (2020). Conservative agency via attainable utility preservation. In *Proceedings of the AAAI/ACM Conference on AI, Ethics, and Society (AIES '20)* (pp. 385-391). ACM.

[7] Kose, U., & Vasant, P. (2017). Fading intelligence theory: A theory on keeping artificial intelligence safety for the future. In *2017 International Artificial Intelligence and Data Processing Symposium (IDAP)*. IEEE. <https://doi.org/10.1109/IDAP.2017.8090235>